# 03 · Feature selection & PCA (RQ2)
Correlation filter, mutual information, RF importance, ANOVA SelectKBest, and PCA — each fitted on the train split and **benchmarked** with a Random Forest.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "ml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, json
print("project root:", ROOT)

In [ ]:
from ml.feature_engineering.selection import FeatureSelector
from ml.feature_engineering.dimensionality_reduction import PCAReducer
from sklearn.ensemble import RandomForestClassifier
from ml.training.evaluate import evaluate_model
import time
s = joblib.load("ml/data/splits/dataset_splits.joblib")
X_train, y_train, X_test, y_test, names = s["X_train"], s["y_train"], s["X_test"], s["y_test"], s["feature_names"]
fs = FeatureSelector(random_state=42)
subsets = {
    "all": names,
    "corr<0.90": fs.correlation_filter(s["X_train_df_raw"], threshold=0.90),
    "MI top25": fs.mutual_information(X_train, y_train, names, top_k=25),
    "RF top25": fs.tree_importance(X_train, y_train, names, top_k=25),
    "ANOVA k25": fs.select_k_best(X_train, y_train, names, k=25),
}
{k: len(v) for k, v in subsets.items()}

In [ ]:
idx = {n: i for i, n in enumerate(names)}
rows = []
for name, feats in subsets.items():
    cols = [idx[f] for f in feats]
    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced", n_jobs=-1)
    t = time.perf_counter(); rf.fit(X_train[:, cols], y_train); tt = time.perf_counter() - t
    ev = evaluate_model(rf, X_test[:, cols], y_test, name, training_time=tt)
    rows.append({"feature_set": name, "n": len(cols), "f1": ev["f1"], "recall": ev["recall"], "train_s": round(tt, 2)})
pca = PCAReducer(variance_threshold=0.95, random_state=42)
Xp_tr, Xp_te = pca.fit_transform(X_train), pca.transform(X_test)
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced", n_jobs=-1)
t = time.perf_counter(); rf.fit(Xp_tr, y_train); tt = time.perf_counter() - t
ev = evaluate_model(rf, Xp_te, y_test, "pca", training_time=tt)
rows.append({"feature_set": "PCA 95%", "n": pca.n_components_retained_, "f1": ev["f1"], "recall": ev["recall"], "train_s": round(tt, 2)})
pd.DataFrame(rows)

In [ ]:
plt.plot(np.arange(1, len(pca.cumulative_variance_[:40]) + 1), pca.cumulative_variance_[:40], marker="o", color="#0072B2")
plt.axhline(0.95, ls="--", color="#D55E00"); plt.xlabel("components"); plt.ylabel("cumulative explained variance"); plt.title("PCA scree"); plt.tight_layout()

In [ ]:
pd.Series(fs.feature_scores_["tree_importance"]).sort_values(ascending=False).head(15)